# ArrowFin: análisis de sesiones de trading

Notebook principal de la prueba. Corre de principio a fin con *Restart & Run All*.

**Secciones**
1. Configuración y carga de datos
2. Chequeos de integridad
3. Enriquecer los fills (trader, valor por punto, sesión)
4. PnL realizado con FIFO
5. PnL no realizado de las posiciones abiertas al corte
6. Tabla de PnL por trader y sesión

## 1. Configuración y carga de datos

Los datos están tal como vienen en `dataset/`, sin modificar. El "ahora" del dataset es **2026-08-25 14:30 UTC**.

In [1]:
from collections import deque
from pathlib import Path

import pandas as pd

def find_project_root():
    """Busca la carpeta que contiene dataset/fills.csv, para que el notebook funcione
    aunque Jupyter o VS Code arranquen en otra carpeta."""
    start = Path.cwd().resolve()
    candidates = [start, *start.parents, *[p.parent for p in start.glob("*/dataset")]]
    for folder in candidates:
        if (folder / "dataset" / "fills.csv").exists():
            return folder
    raise FileNotFoundError(f"No encuentro dataset/fills.csv desde {start}. "
                            "Abre el notebook desde la carpeta del proyecto.")


ROOT = find_project_root()
DATA = ROOT / "dataset"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)
print("Carpeta del proyecto:", ROOT.name)

CUT = pd.Timestamp("2026-08-25T14:30:00Z")  # "ahora" según dataset/README.md

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

Carpeta del proyecto: PruebTecnicaArrowFin


In [2]:
fills = pd.read_csv(DATA / "fills.csv", parse_dates=["filled_at"])
accounts = pd.read_csv(DATA / "accounts.csv")
traders = pd.read_csv(DATA / "traders.csv")
instruments = pd.read_csv(DATA / "instruments.csv")
prices = pd.read_csv(DATA / "market_prices.csv")

for name, df in [("fills", fills), ("accounts", accounts), ("traders", traders),
                 ("instruments", instruments), ("prices", prices)]:
    print(f"{name:12s} {df.shape}")

fills        (5146, 10)
accounts     (54, 9)
traders      (48, 8)
instruments  (8, 8)
prices       (8, 3)


In [3]:
fills.head()

,id,account_id,instrument_symbol,side,quantity,price,filled_at,order_id,liquidity,commission_usd
0,FIL-600001,ACC-2037,MES,BUY,3,5609.00,2026-08-10 10:34:12+00:00,ORD-903693,taker,2.1
1,FIL-600002,ACC-2011,MNQ,SELL,3,19726.50,2026-08-10 10:34:33+00:00,ORD-901143,taker,2.1
2,FIL-600003,ACC-2002,ES,BUY,6,5609.25,2026-08-10 10:34:52+00:00,ORD-900137,taker,12.6
3,FIL-600004,ACC-2024,MNQ,BUY,3,19726.50,2026-08-10 10:35:07+00:00,ORD-902894,taker,2.1
4,FIL-600005,ACC-2017,MES,BUY,3,5607.75,2026-08-10 10:35:24+00:00,ORD-902525,taker,2.1


In [4]:
instruments[["symbol", "description", "point_value_usd", "tick_size", "tick_value_usd"]]

,symbol,description,point_value_usd,tick_size,tick_value_usd
0,ES,E-mini S&P 500,50,0.25000,12.50
1,MES,Micro E-mini S&P 500,5,0.25000,1.25
2,NQ,E-mini Nasdaq-100,20,0.25000,5.00
3,MNQ,Micro E-mini Nasdaq-100,2,0.25000,0.50
4,CL,Crude Oil,1000,0.01000,10.00
5,MCL,Micro Crude Oil,100,0.01000,1.00
6,GC,Gold,100,0.10000,10.00
7,6E,Euro FX,125000,0.00005,6.25


## 2. Chequeos de integridad

Antes de calcular nada, confirmamos que los datos no tengan problemas básicos. Si alguno falla, el `assert` detiene el notebook.

In [5]:
assert fills.isna().sum().sum() == 0, "hay nulos en fills"
assert not fills["id"].duplicated().any(), "ids de fill duplicados"
assert (fills["quantity"] > 0).all() and (fills["price"] > 0).all(), "cantidades o precios no positivos"
assert (fills["filled_at"] <= CUT).all(), "hay fills posteriores al corte"
assert fills["account_id"].isin(accounts["id"]).all(), "fills de cuentas que no existen"
assert fills["instrument_symbol"].isin(instruments["symbol"]).all(), "instrumentos desconocidos"
assert (fills.groupby("order_id")["account_id"].nunique() == 1).all(), "una orden en varias cuentas"

print("Fills:", len(fills), "| órdenes:", fills["order_id"].nunique(),
      "| órdenes con más de un fill:", (fills.groupby("order_id").size() > 1).sum())
print("Rango:", fills["filled_at"].min(), "->", fills["filled_at"].max())

Fills: 5146 | órdenes: 4886 | órdenes con más de un fill: 260
Rango: 2026-08-10 10:34:12+00:00 -> 2026-08-25 14:19:39+00:00


**Comisiones.** Son constantes por contrato y por lado en cada instrumento, así que `commission_usd` ya es el costo total del fill y solo hay que sumarlo.

In [6]:
(fills["commission_usd"] / fills["quantity"]).groupby(fills["instrument_symbol"]).agg(["min", "max"])

,min,max
instrument_symbol,,
6E,2.48,2.48
CL,2.30,2.30
ES,2.10,2.10
GC,2.10,2.10
MCL,0.70,0.70
MES,0.70,0.70
MNQ,0.70,0.70
NQ,2.10,2.10


**Cuentas con estado especial.** No se eliminan: se dejan anotadas para revisarlas en el ranking.

In [7]:
special = accounts[(accounts["status"] != "active") | (accounts["balance"] <= 0)]
special.assign(n_fills=special["id"].map(fills["account_id"].value_counts()).fillna(0).astype(int))[
    ["id", "trader_id", "account_type", "balance", "status", "n_fills"]]

,id,trader_id,account_type,balance,status,n_fills
27,ACC-2028,T-025,funded,0.0,active,3
52,ACC-2053,T-040,eval,31500.0,closed,20
53,ACC-2054,T-041,funded,25000.0,restricted,3


## 3. Enriquecer los fills

- **Trader:** cada cuenta pertenece a un trader, y el análisis se hace por trader.
- **Valor por punto:** un punto de NQ vale $20 y uno de MNQ vale $2. Sin esto, el PnL sale mal.
- **Cantidad con signo:** compra = +, venta = −.
- **Sesión:** la sesión de CME va de 17:00 a 16:00 hora de Chicago. Sumando 7 horas a la hora de Chicago,
  la apertura (17:00) pasa a medianoche del día siguiente, así que la fecha resultante es la del día en que cierra la sesión.

In [8]:
fills = (
    fills
    .merge(accounts[["id", "trader_id"]].rename(columns={"id": "account_id"}),
           on="account_id", how="left", validate="many_to_one")
    .merge(instruments[["symbol", "point_value_usd"]].rename(columns={"symbol": "instrument_symbol"}),
           on="instrument_symbol", how="left", validate="many_to_one")
)
fills["signed_qty"] = fills["quantity"].where(fills["side"] == "BUY", -fills["quantity"])

chicago_time = fills["filled_at"].dt.tz_convert("America/Chicago").dt.tz_localize(None)
fills["session_date"] = (chicago_time + pd.Timedelta(hours=7)).dt.strftime("%Y-%m-%d")

assert fills[["trader_id", "point_value_usd"]].notna().all().all()
fills[["id", "account_id", "trader_id", "instrument_symbol", "signed_qty", "price",
       "point_value_usd", "filled_at", "session_date"]].head()

,id,account_id,trader_id,instrument_symbol,signed_qty,price,point_value_usd,filled_at,session_date
0,FIL-600001,ACC-2037,T-034,MES,3,5609.00,5,2026-08-10 10:34:12+00:00,2026-08-10
1,FIL-600002,ACC-2011,T-011,MNQ,-3,19726.50,2,2026-08-10 10:34:33+00:00,2026-08-10
2,FIL-600003,ACC-2002,T-002,ES,6,5609.25,50,2026-08-10 10:34:52+00:00,2026-08-10
3,FIL-600004,ACC-2024,T-021,MNQ,3,19726.50,2,2026-08-10 10:35:07+00:00,2026-08-10
4,FIL-600005,ACC-2017,T-017,MES,3,5607.75,5,2026-08-10 10:35:24+00:00,2026-08-10


Fills en los que la fecha UTC **no** coincide con la sesión. Si agrupáramos por día UTC, estos quedarían en la sesión equivocada.

In [9]:
utc_date = fills["filled_at"].dt.strftime("%Y-%m-%d")
wrong_day = fills[utc_date != fills["session_date"]]
print("Fills cuya fecha UTC no es su sesión:", len(wrong_day))
wrong_day[["id", "trader_id", "filled_at", "session_date"]].head(10)

Fills cuya fecha UTC no es su sesión: 10


,id,trader_id,filled_at,session_date
4649,FIL-604650,T-020,2026-08-24 22:02:45+00:00,2026-08-25
4650,FIL-604651,T-021,2026-08-24 22:05:42+00:00,2026-08-25
4651,FIL-604652,T-020,2026-08-24 22:14:45+00:00,2026-08-25
4652,FIL-604653,T-021,2026-08-24 22:15:42+00:00,2026-08-25
4653,FIL-604654,T-021,2026-08-24 22:23:21+00:00,2026-08-25
4654,FIL-604655,T-021,2026-08-24 22:26:21+00:00,2026-08-25
4655,FIL-604656,T-020,2026-08-24 22:46:34+00:00,2026-08-25
4656,FIL-604657,T-020,2026-08-24 23:02:34+00:00,2026-08-25
4657,FIL-604658,T-020,2026-08-24 23:04:23+00:00,2026-08-25
4658,FIL-604659,T-020,2026-08-24 23:06:23+00:00,2026-08-25


Primera y última operación de cada sesión, en hora de Chicago. **La sesión de hoy está incompleta:** el corte es a las 09:30 CT
y las sesiones anteriores tienen actividad hasta alrededor de las 15:20 CT. Por eso, el resultado de hoy es "lo que va de la sesión".

In [10]:
fills["chicago_time"] = chicago_time
fills.groupby("session_date").agg(fills=("id", "size"), traders=("trader_id", "nunique"),
                                  first_fill_ct=("chicago_time", "min"), last_fill_ct=("chicago_time", "max"))

,fills,traders,first_fill_ct,last_fill_ct
session_date,,,,
2026-08-10,309,28,2026-08-10 05:34:12,2026-08-10 15:22:00
2026-08-11,365,35,2026-08-11 05:34:03,2026-08-11 15:16:19
2026-08-12,426,44,2026-08-12 05:34:08,2026-08-12 15:22:42
2026-08-13,452,45,2026-08-13 05:34:00,2026-08-13 15:20:32
2026-08-14,446,45,2026-08-14 05:34:09,2026-08-14 15:21:34
2026-08-17,422,45,2026-08-17 05:34:14,2026-08-17 15:23:44
2026-08-18,441,45,2026-08-18 05:34:03,2026-08-18 15:23:37
2026-08-19,424,45,2026-08-19 05:34:30,2026-08-19 15:21:17
2026-08-20,454,45,2026-08-20 05:35:29,2026-08-20 15:19:12


## 4. PnL realizado con FIFO

Para cada cuenta e instrumento recorremos los fills en orden de tiempo y guardamos los lotes abiertos.
Cuando llega un fill del lado contrario, cierra primero el lote **más antiguo** (FIFO):

`PnL = (precio de salida − precio de entrada) × contratos cerrados × valor por punto`

El PnL se asigna al fill que **cierra** la posición, así que cuenta en la sesión en la que se cerró.
Un fill puede cerrar una posición y abrir otra en sentido contrario (por ejemplo, largo 2 y vende 5: cierra 2 y queda corto 3).

In [11]:
def fifo_realized(group):
    """Recorre los fills de una cuenta e instrumento en orden.
    Devuelve el PnL realizado de cada fill y los lotes que quedan abiertos al final."""
    lots = deque()  # cada lote: [cantidad con signo, precio de entrada, hora de entrada]
    realized = []
    for qty, price, point_value, time in zip(group["signed_qty"], group["price"],
                                             group["point_value_usd"], group["filled_at"]):
        pnl = 0.0
        # Mientras el fill vaya en sentido contrario al lote más antiguo, lo va cerrando.
        while qty != 0 and lots and (lots[0][0] > 0) != (qty > 0):
            lot = lots[0]
            closed = min(abs(qty), abs(lot[0]))
            lot_direction = 1 if lot[0] > 0 else -1  # +1 largo, -1 corto
            pnl += (price - lot[1]) * closed * lot_direction * point_value
            lot[0] -= closed * lot_direction
            qty += closed * lot_direction
            if lot[0] == 0:
                lots.popleft()
        if qty != 0:  # lo que sobra abre (o amplía) una posición
            lots.append([qty, price, time])
        realized.append(pnl)
    return realized, list(lots)

Prueba rápida con un caso a mano: compra 2 MNQ a 100, vende 5 a 110 (cierra 2 y abre un corto de 3) y compra 3 a 105.
Se espera +$40 en el segundo fill (10 pts × 2 × $2) y +$30 en el tercero (5 pts × 3 × $2).

In [12]:
toy = pd.DataFrame({"signed_qty": [2, -5, 3], "price": [100.0, 110.0, 105.0],
                    "point_value_usd": [2, 2, 2], "filled_at": [1, 2, 3]})
toy_realized, toy_lots = fifo_realized(toy)
print(toy_realized, toy_lots)
assert toy_realized == [0.0, 40.0, 30.0] and toy_lots == []

[0.0, 40.0, 30.0] []


In [13]:
fills = fills.sort_values(["filled_at", "id"]).reset_index(drop=True)
fills["realized_pnl"] = 0.0
open_lots = []

for (account_id, symbol), group in fills.groupby(["account_id", "instrument_symbol"]):
    realized, lots = fifo_realized(group)
    fills.loc[group.index, "realized_pnl"] = realized
    for qty, entry_price, entry_time in lots:
        open_lots.append({"account_id": account_id, "instrument_symbol": symbol,
                          "qty": qty, "entry_price": entry_price, "entry_time": entry_time})

open_lots = pd.DataFrame(open_lots)
print("PnL realizado total: $", round(fills["realized_pnl"].sum(), 2))

PnL realizado total: $ -43209.5


**Posiciones que pasan de una sesión a otra.** Si una posición sigue abierta al cierre de una sesión, su PnL se registra en la sesión en que se cierra.
Así, el movimiento de precio de la noche cae en la sesión siguiente. Es una simplificación: lo correcto sería
valorar al precio de *settlement* de cada día, pero el dataset no lo trae. Contamos cuántas veces ocurre para saber cuánto pesa.

In [14]:
position_end_of_session = (
    fills.assign(position=fills.groupby(["account_id", "instrument_symbol"])["signed_qty"].cumsum())
    .groupby(["account_id", "instrument_symbol", "session_date"])["position"].last()
)
carried = position_end_of_session[position_end_of_session != 0]
print("Cierres de sesión con posición abierta:", len(carried),
      "de", len(position_end_of_session), "combinaciones cuenta-instrumento-sesión")
carried.reset_index().head(10)

Cierres de sesión con posición abierta: 8 de 558 combinaciones cuenta-instrumento-sesión


,account_id,instrument_symbol,session_date,position
0,ACC-2017,MES,2026-08-25,6
1,ACC-2026,CL,2026-08-25,-3
2,ACC-2028,MNQ,2026-08-25,1
3,ACC-2034,MES,2026-08-25,1
4,ACC-2039,MES,2026-08-25,2
5,ACC-2044,MES,2026-08-25,2
6,ACC-2050,MES,2026-08-25,1
7,ACC-2054,MNQ,2026-08-25,2


## 5. PnL no realizado al corte

Las posiciones que siguen abiertas al corte se valoran con el precio de `market_prices.csv`.
Todo el no realizado se asigna a la sesión de hoy, porque es la que está en curso.

In [15]:
TODAY = fills["session_date"].max()

open_lots = (
    open_lots
    .merge(prices[["symbol", "mark_price"]].rename(columns={"symbol": "instrument_symbol"}),
           on="instrument_symbol", how="left", validate="many_to_one")
    .merge(instruments[["symbol", "point_value_usd"]].rename(columns={"symbol": "instrument_symbol"}),
           on="instrument_symbol", how="left", validate="many_to_one")
    .merge(accounts[["id", "trader_id"]].rename(columns={"id": "account_id"}),
           on="account_id", how="left", validate="many_to_one")
)
open_lots["unrealized_pnl"] = (
    (open_lots["mark_price"] - open_lots["entry_price"]) * open_lots["qty"] * open_lots["point_value_usd"]
)
open_lots["session_date"] = TODAY
print("Sesión de hoy:", TODAY)
open_lots

Sesión de hoy: 2026-08-25


,account_id,instrument_symbol,qty,entry_price,entry_time,mark_price,point_value_usd,trader_id,unrealized_pnl,session_date
0,ACC-2017,MES,3,5640.00,2026-08-25 14:04:28+00:00,5642.25,5,T-017,33.75,2026-08-25
1,ACC-2017,MES,3,5640.00,2026-08-25 14:04:29+00:00,5642.25,5,T-017,33.75,2026-08-25
2,ACC-2026,CL,-3,73.40,2026-08-25 12:50:19+00:00,74.18,1000,T-023,-2340.00,2026-08-25
3,ACC-2028,MNQ,1,19833.50,2026-08-25 14:02:23+00:00,19845.50,2,T-025,24.00,2026-08-25
4,ACC-2034,MES,1,5640.75,2026-08-25 14:04:19+00:00,5642.25,5,T-031,7.50,2026-08-25
5,ACC-2039,MES,2,5640.75,2026-08-25 14:04:52+00:00,5642.25,5,T-036,15.00,2026-08-25
6,ACC-2044,MES,2,5640.75,2026-08-25 14:03:17+00:00,5642.25,5,T-041,15.00,2026-08-25
7,ACC-2050,MES,1,5640.75,2026-08-25 14:00:03+00:00,5642.25,5,T-046,7.50,2026-08-25
8,ACC-2054,MNQ,2,19805.50,2026-08-25 14:02:58+00:00,19845.50,2,T-041,160.00,2026-08-25


**Chequeo de cuadre.** El PnL total también se puede calcular sin FIFO, como flujo de caja:
lo que se cobró por las ventas, menos lo que se pagó por las compras, más lo que valen hoy las posiciones abiertas.
Los dos cálculos tienen que coincidir.

In [16]:
cash_flow = (-fills["signed_qty"] * fills["price"] * fills["point_value_usd"]).sum()
final_position = fills.groupby("instrument_symbol")["signed_qty"].sum()
open_value = (final_position
              * prices.set_index("symbol")["mark_price"].reindex(final_position.index)
              * instruments.set_index("symbol")["point_value_usd"].reindex(final_position.index)).sum()

fifo_total = fills["realized_pnl"].sum() + open_lots["unrealized_pnl"].sum()
print(f"FIFO: {fifo_total:,.2f}  |  flujo de caja: {cash_flow + open_value:,.2f}")
assert abs(fifo_total - (cash_flow + open_value)) < 0.01

FIFO: -45,253.00  |  flujo de caja: -45,253.00


## 6. Tabla de PnL por trader y sesión

Una fila por trader y sesión:

| columna | qué es |
|---|---|
| `realized_pnl` | PnL de las posiciones cerradas en la sesión |
| `unrealized_pnl` | PnL de las posiciones abiertas al corte (solo hoy) |
| `commissions` | comisiones pagadas |
| `net_pnl` | realizado + no realizado − comisiones |
| `contracts` | contratos operados (compras + ventas) |
| `orders` | órdenes (una orden partida en varios fills cuenta una vez) |
| `volume_usd_per_point` | contratos × valor por punto: tamaño operado comparable entre instrumentos (1 NQ = 10 MNQ) |
| `instruments` | instrumentos operados |

In [17]:
pnl = (
    fills
    .assign(volume_usd_per_point=fills["quantity"] * fills["point_value_usd"])
    .groupby(["trader_id", "session_date"])
    .agg(realized_pnl=("realized_pnl", "sum"),
         commissions=("commission_usd", "sum"),
         contracts=("quantity", "sum"),
         orders=("order_id", "nunique"),
         fills=("id", "size"),
         volume_usd_per_point=("volume_usd_per_point", "sum"),
         instruments=("instrument_symbol", lambda s: ",".join(sorted(s.unique()))))
)

unrealized = open_lots.groupby(["trader_id", "session_date"])["unrealized_pnl"].sum()
pnl = pnl.join(unrealized, how="left").fillna({"unrealized_pnl": 0.0}).reset_index()

pnl["net_pnl"] = pnl["realized_pnl"] + pnl["unrealized_pnl"] - pnl["commissions"]
pnl = pnl[["trader_id", "session_date", "net_pnl", "realized_pnl", "unrealized_pnl", "commissions",
           "contracts", "orders", "fills", "volume_usd_per_point", "instruments"]].round(2)

assert not pnl.duplicated(["trader_id", "session_date"]).any()
print(pnl.shape)
pnl.head(12)

(516, 11)


,trader_id,session_date,net_pnl,realized_pnl,unrealized_pnl,commissions,contracts,orders,fills,volume_usd_per_point,instruments
0,T-001,2026-08-10,-628.20,-435.00,0.0,193.2,92,12,12,1840,NQ
1,T-001,2026-08-11,-664.80,-480.00,0.0,184.8,88,10,12,1760,NQ
2,T-001,2026-08-12,-2226.00,-1995.00,0.0,231.0,110,14,14,2200,NQ
3,T-001,2026-08-13,2319.80,2450.00,0.0,130.2,62,10,11,3100,ES
4,T-001,2026-08-14,-3010.10,-2787.50,0.0,222.6,106,14,15,5300,ES
5,T-001,2026-08-17,1997.55,2043.75,0.0,46.2,66,8,10,330,MES
6,T-001,2026-08-18,-2589.10,-2387.50,0.0,201.6,96,14,15,4800,ES
7,T-001,2026-08-19,2140.40,2300.00,0.0,159.6,76,12,13,3800,ES
8,T-001,2026-08-20,118.00,265.00,0.0,147.0,70,8,8,1400,NQ
9,T-001,2026-08-21,970.40,1130.00,0.0,159.6,76,12,14,1520,NQ


Chequeos rápidos: el total por sesión y cuánto historial tiene cada trader.

In [18]:
pnl.groupby("session_date")[["net_pnl", "realized_pnl", "unrealized_pnl", "commissions"]].sum().round(0)

,net_pnl,realized_pnl,unrealized_pnl,commissions
session_date,,,,
2026-08-10,-6269.0,-4676.0,0.0,1593.0
2026-08-11,-8805.0,-7002.0,0.0,1803.0
2026-08-12,3200.0,5146.0,0.0,1946.0
2026-08-13,-3400.0,-1654.0,0.0,1746.0
2026-08-14,-12911.0,-10943.0,0.0,1968.0
2026-08-17,3284.0,4968.0,0.0,1684.0
2026-08-18,1374.0,3099.0,0.0,1725.0
2026-08-19,1264.0,3052.0,0.0,1788.0
2026-08-20,-3107.0,-1026.0,0.0,2081.0


In [19]:
sessions_per_trader = pnl.groupby("trader_id")["session_date"].nunique()
print("Sesiones por trader (cuántos traders tienen N sesiones):")
print(sessions_per_trader.value_counts().sort_index().to_string())
print("\nTraders con menos de 5 sesiones:",
      sessions_per_trader[sessions_per_trader < 5].to_dict())

Sesiones por trader (cuántos traders tienen N sesiones):
session_date
1      2
2      1
9      1
10     9
11     7
12    28

Traders con menos de 5 sesiones: {'T-024': 1, 'T-025': 1, 'T-026': 2}


Ejemplo: el historial completo de T-020, el trader de la Tarea 5.

In [20]:
pnl[pnl["trader_id"] == "T-020"]

,trader_id,session_date,net_pnl,realized_pnl,unrealized_pnl,commissions,contracts,orders,fills,volume_usd_per_point,instruments
223,T-020,2026-08-10,-137.10,-117.50,0.0,19.6,28,6,6,140,MES
224,T-020,2026-08-11,-1267.40,-1245.00,0.0,22.4,32,6,7,160,MES
225,T-020,2026-08-12,-44.15,-28.75,0.0,15.4,22,6,6,110,MES
226,T-020,2026-08-13,110.40,130.00,0.0,19.6,28,6,6,140,MES
227,T-020,2026-08-14,17.45,41.25,0.0,23.8,34,6,7,170,MES
228,T-020,2026-08-17,678.05,696.25,0.0,18.2,26,6,7,130,MES
229,T-020,2026-08-18,-387.75,-373.75,0.0,14.0,20,6,6,100,MES
230,T-020,2026-08-19,342.75,363.75,0.0,21.0,30,6,7,150,MES
231,T-020,2026-08-20,285.40,305.00,0.0,19.6,28,6,6,140,MES
232,T-020,2026-08-21,-769.00,-755.00,0.0,14.0,20,6,6,100,MES


In [21]:
pnl.to_csv(OUT / "pnl_sessions.csv", index=False)
print("Guardado: output/pnl_sessions.csv")

Guardado: output/pnl_sessions.csv


## 7. Precios sospechosos

Algunos fills de MCL se alejan bastante del resto del mercado en la misma sesión. Por ejemplo, el 24/08 hay un fill a 79.29
cuando la mediana del día es 74.14. No podemos saber si son errores del feed o ejecuciones reales a mal precio,
así que **no los eliminamos**: los marcamos y revisamos si afectan a algún trader del ranking.

Criterio: el precio se aleja más de un 2% de la mediana de ese instrumento en esa sesión.

In [22]:
session_median = fills.groupby(["instrument_symbol", "session_date"])["price"].transform("median")
fills["price_deviation"] = (fills["price"] / session_median - 1).abs()
suspicious = fills[fills["price_deviation"] > 0.02]

print("Fills con precio a más de 2% de la mediana de su sesión:", len(suspicious))
suspicious[["id", "trader_id", "account_id", "instrument_symbol", "side", "quantity", "price",
            "session_date", "realized_pnl"]].assign(deviation_pct=(suspicious["price_deviation"] * 100).round(1))

Fills con precio a más de 2% de la mediana de su sesión: 9


,id,trader_id,account_id,instrument_symbol,side,quantity,price,session_date,realized_pnl,deviation_pct
157,FIL-600158,T-023,ACC-2026,MCL,BUY,2,75.30,2026-08-10,-316.0,2.2
359,FIL-600360,T-047,ACC-2051,MCL,SELL,2,71.71,2026-08-11,-420.0,2.8
646,FIL-600647,T-047,ACC-2051,MCL,SELL,1,69.60,2026-08-11,-418.0,5.7
1603,FIL-601604,T-047,ACC-2051,MCL,SELL,1,72.39,2026-08-14,-147.0,2.0
2040,FIL-602041,T-048,ACC-2052,MCL,BUY,1,77.20,2026-08-17,-328.0,4.4
2387,FIL-602388,T-048,ACC-2052,MCL,BUY,1,76.46,2026-08-17,-252.0,3.4
3142,FIL-603143,T-042,ACC-2045,MCL,SELL,1,75.79,2026-08-19,178.0,2.4
4238,FIL-604239,T-030,ACC-2033,MCL,SELL,1,79.29,2026-08-24,518.0,6.9
4619,FIL-604620,T-030,ACC-2033,MCL,SELL,2,76.35,2026-08-24,438.0,3.0


## 8. Tarea 1: top 10 de traders con una sesión inusualmente mala

**Definición.** Una sesión es *inusualmente mala* cuando el trader pierde dinero y esa pérdida está muy por debajo de lo que
**él mismo** suele hacer. Lo medimos con un z-score:

`z = (PnL neto de hoy − su PnL medio en sesiones anteriores) / su desviación estándar en sesiones anteriores`

- `z = −1`: un mal día normal para ese trader.
- `z = −3`: una sesión tres veces más mala de lo que suele variar su resultado.

**Por qué así.** Un trader que opera 10 contratos tiene sesiones que varían mucho más, en dólares, que uno que opera 1.
Dividir por su propia variabilidad hace que $2,000 pesen poco para el primero y mucho para el segundo, que es lo que pide el enunciado.

**Reglas**
1. Se evalúa la sesión de **hoy** (2026-08-25), que es la que el revisor verá mañana.
2. La base usa solo sesiones **anteriores** a hoy. Hoy nunca entra en su propia base.
3. Solo entran sesiones con PnL neto negativo: una sesión que gana dinero no es "mala".
4. Traders con menos de 5 sesiones previas: su z no es fiable y se listan aparte.

In [23]:
MIN_PRIOR_SESSIONS = 5

history = pnl[pnl["session_date"] < TODAY]
today = pnl[pnl["session_date"] == TODAY]

baseline = history.groupby("trader_id").agg(
    prior_sessions=("session_date", "size"),
    mean_pnl=("net_pnl", "mean"),
    std_pnl=("net_pnl", "std"),
    worst_prior_pnl=("net_pnl", "min"),
    usual_volume=("volume_usd_per_point", "median"),
)

scored = today.merge(baseline, on="trader_id", how="left")
scored["prior_sessions"] = scored["prior_sessions"].fillna(0).astype(int)
scored["z"] = (scored["net_pnl"] - scored["mean_pnl"]) / scored["std_pnl"]
scored["volume_vs_usual"] = scored["volume_usd_per_point"] / scored["usual_volume"]
scored["worse_than_any_prior"] = scored["net_pnl"] < scored["worst_prior_pnl"]

assert (scored["std_pnl"].dropna() > 0).all(), "desviación cero: el z no se puede calcular"
print("Traders con sesión hoy:", len(scored), "| con pérdida:", (scored["net_pnl"] < 0).sum())

Traders con sesión hoy: 48 | con pérdida: 38


Contexto de cada trader para el revisor: si tiene alguna cuenta con estado especial, fills con precio sospechoso hoy o posiciones abiertas al corte.

In [24]:
special_traders = set(special["trader_id"])
suspicious_today = set(suspicious.loc[suspicious["session_date"] == TODAY, "trader_id"])
open_today = set(open_lots["trader_id"])

def notes(trader_id):
    out = []
    if trader_id in special_traders:
        out.append("cuenta con estado especial")
    if trader_id in suspicious_today:
        out.append("precio sospechoso hoy")
    if trader_id in open_today:
        out.append("posición abierta al corte")
    return "; ".join(out)

scored["notes"] = scored["trader_id"].map(notes)

In [25]:
eligible = scored[(scored["prior_sessions"] >= MIN_PRIOR_SESSIONS) & (scored["net_pnl"] < 0)]

top10 = eligible.nsmallest(10, "z").reset_index(drop=True)
top10.index = top10.index + 1
top10.index.name = "rank"

report_columns = ["trader_id", "net_pnl", "mean_pnl", "std_pnl", "z", "worst_prior_pnl", "worse_than_any_prior",
                  "contracts", "volume_vs_usual", "instruments", "prior_sessions", "notes"]
top10[report_columns].round(2)

,trader_id,net_pnl,mean_pnl,std_pnl,z,worst_prior_pnl,worse_than_any_prior,contracts,volume_vs_usual,instruments,prior_sessions,notes
rank,,,,,,,,,,,,
1,T-018,-1516.80,-110.14,157.81,-8.91,-350.85,True,24,0.86,MES,8,
2,T-023,-2486.10,-377.89,387.07,-5.45,-919.40,True,19,2.30,"CL,MCL",11,posición abierta al corte
3,T-019,-1559.20,-9.03,305.19,-5.08,-494.50,True,26,1.08,MNQ,9,
4,T-009,-669.95,-66.89,121.31,-4.97,-252.30,True,12,3.75,"ES,MES",11,
5,T-011,-758.55,8.09,165.86,-4.62,-246.10,True,14,2.33,MES,11,
6,T-012,-724.10,21.87,174.18,-4.28,-320.20,True,18,1.00,MNQ,11,
7,T-020,-2397.65,-100.20,541.69,-4.24,-1267.40,True,52,1.86,MES,11,
8,T-010,-663.85,-61.33,179.58,-3.36,-359.80,True,18,3.75,MES,11,
9,T-008,-459.00,-29.35,151.29,-2.84,-208.40,True,10,0.40,MNQ,11,


### Traders sin historial suficiente

No entran al ranking porque su z no es fiable, pero se muestran para que el revisor los tenga en cuenta.

In [26]:
scored[scored["prior_sessions"] < MIN_PRIOR_SESSIONS][
    ["trader_id", "net_pnl", "prior_sessions", "contracts", "instruments", "notes"]]

,trader_id,net_pnl,prior_sessions,contracts,instruments,notes
23,T-024,-1579.60,0,38,MNQ,
24,T-025,-72.00,0,5,"MES,MNQ",cuenta con estado especial; posición abierta a...
25,T-026,-127.95,1,6,MES,


### Chequeos del ranking

**¿Qué cambia si ordenamos solo por pérdida en dólares?** Si el top 10 fuera igual, normalizar no aportaría nada.

In [27]:
top10_by_dollars = eligible.nsmallest(10, "net_pnl")["trader_id"]
overlap = set(top10["trader_id"]) & set(top10_by_dollars)
print(f"Coinciden {len(overlap)} de 10")
print("Solo en el ranking por z:     ", sorted(set(top10["trader_id"]) - set(top10_by_dollars)))
print("Solo en el ranking por dólares:", sorted(set(top10_by_dollars) - set(top10["trader_id"])))

Coinciden 3 de 10
Solo en el ranking por z:      ['T-008', 'T-009', 'T-010', 'T-011', 'T-012', 'T-013', 'T-018']
Solo en el ranking por dólares: ['T-001', 'T-002', 'T-003', 'T-004', 'T-005', 'T-006', 'T-007']


**¿Es estable si usamos mediana en lugar de media?** Con unas 10 sesiones previas, una sola sesión extrema mueve bastante la media y la desviación.
La versión robusta usa mediana y MAD (desviación absoluta mediana, escalada ×1.4826 para que sea comparable a la desviación estándar).

In [28]:
robust = history.groupby("trader_id")["net_pnl"].agg(
    median_pnl="median",
    mad_pnl=lambda s: 1.4826 * (s - s.median()).abs().median(),
)
eligible_robust = eligible.merge(robust, on="trader_id")
eligible_robust["z_robust"] = (eligible_robust["net_pnl"] - eligible_robust["median_pnl"]) / eligible_robust["mad_pnl"]
top10_robust = eligible_robust.nsmallest(10, "z_robust")["trader_id"]

print(f"Coinciden {len(set(top10['trader_id']) & set(top10_robust))} de 10 con la versión robusta")
print("Top 10 (media/desv.):  ", list(top10["trader_id"]))
print("Top 10 (mediana/MAD):  ", list(top10_robust))

Coinciden 8 de 10 con la versión robusta
Top 10 (media/desv.):   ['T-018', 'T-023', 'T-019', 'T-009', 'T-011', 'T-012', 'T-020', 'T-010', 'T-008', 'T-013']
Top 10 (mediana/MAD):   ['T-011', 'T-018', 'T-019', 'T-009', 'T-020', 'T-012', 'T-023', 'T-044', 'T-008', 'T-029']


In [29]:
top10[report_columns].round(2).to_csv(OUT / "top10.csv")
print("Guardado: output/top10.csv")

Guardado: output/top10.csv


## 9. Tarea 5: revisión de T-020

T-020 dice que operó "exactamente como siempre". Antes de responder, comparamos su sesión de hoy con su historial en tres aspectos:
tamaño de cada orden, cantidad de órdenes y horario.

In [30]:
t020 = fills[fills["trader_id"] == "T-020"]
order_size = t020.groupby(["session_date", "order_id"])["quantity"].sum()

t020_sessions = t020.groupby("session_date").agg(
    orders=("order_id", "nunique"),
    contracts=("quantity", "sum"),
    first_fill_ct=("chicago_time", "min"),
    last_fill_ct=("chicago_time", "max"),
    realized_pnl=("realized_pnl", "sum"),
).join(order_size.groupby("session_date").max().rename("max_order_size"))
t020_sessions

,orders,contracts,first_fill_ct,last_fill_ct,realized_pnl,max_order_size
session_date,,,,,,
2026-08-10,6,28,2026-08-10 05:43:05,2026-08-10 14:55:59,-117.50,5
2026-08-11,6,32,2026-08-11 05:41:47,2026-08-11 14:59:24,-1245.00,6
2026-08-12,6,22,2026-08-12 05:35:38,2026-08-12 14:53:05,-28.75,5
2026-08-13,6,28,2026-08-13 05:42:14,2026-08-13 14:53:10,130.00,6
2026-08-14,6,34,2026-08-14 05:40:47,2026-08-14 14:59:47,41.25,6
2026-08-17,6,26,2026-08-17 05:34:33,2026-08-17 15:06:58,696.25,5
2026-08-18,6,20,2026-08-18 05:37:28,2026-08-18 15:02:48,-373.75,4
2026-08-19,6,30,2026-08-19 05:41:16,2026-08-19 15:09:37,363.75,6
2026-08-20,6,28,2026-08-20 05:41:53,2026-08-20 15:16:24,305.00,6


Fills de hoy. En sus 11 sesiones anteriores, T-020 siempre operó entre las 05:30 y las 15:20 CT.
Hoy, los primeros ocho fills ocurrieron fuera de ese horario: seis en la apertura nocturna de Globex (17:00 a 18:10 CT de la tarde anterior)
y dos a las 02:00 CT.

In [31]:
t020_today = t020[t020["session_date"] == TODAY]
hour = t020_today["chicago_time"].dt.hour
off_hours = (hour >= 17) | (hour < 5)  # fuera de su horario habitual (05:30 a 15:20 CT)

print(f"PnL realizado hoy:              {t020_today['realized_pnl'].sum():>9,.2f}")
print(f"  fuera de su horario habitual:  {t020_today.loc[off_hours, 'realized_pnl'].sum():>9,.2f}")
print(f"  en su horario habitual:        {t020_today.loc[~off_hours, 'realized_pnl'].sum():>9,.2f}")
t020_today[["id", "chicago_time", "side", "quantity", "price", "order_id", "realized_pnl"]]

PnL realizado hoy:              -2,361.25
  fuera de su horario habitual:  -2,076.25
  en su horario habitual:          -285.00


,id,chicago_time,side,quantity,price,order_id,realized_pnl
4649,FIL-604650,2026-08-24 17:02:45,BUY,5,5642.00,ORD-902888,0.00
4651,FIL-604652,2026-08-24 17:14:45,SELL,5,5621.75,ORD-902889,-506.25
4655,FIL-604656,2026-08-24 17:46:34,SELL,6,5642.75,ORD-902886,0.00
4656,FIL-604657,2026-08-24 18:02:34,BUY,6,5662.25,ORD-902887,-585.00
4657,FIL-604658,2026-08-24 18:04:23,BUY,6,5642.00,ORD-902884,0.00
4658,FIL-604659,2026-08-24 18:06:23,SELL,6,5617.50,ORD-902885,-735.00
4710,FIL-604711,2026-08-25 02:04:00,BUY,5,5642.25,ORD-902890,0.00
4725,FIL-604726,2026-08-25 02:11:00,SELL,5,5632.25,ORD-902891,-250.00
5037,FIL-605038,2026-08-25 08:32:28,SELL,2,5641.25,ORD-902892,0.00
5038,FIL-605039,2026-08-25 08:32:29,SELL,2,5641.25,ORD-902892,0.00
